# 05 — Walk-Forward-Tuning (Optuna/TPE)

Treibt `src.backtest.tuning` (`tune_all_baskets`/`tune_basket_params`) end-to-end: läuft eine Walk-Forward-Optuna-Suche pro Basket über den Calibration-Zeitraum, speichert die gefundenen `best_params` unter `data/processed/tuning/` (`src.backtest.tuning.save_tuning_results`) und führt anschliessend EINEN finalen Backtest auf dem Backtest-Zeitraum mit den getunten Parametern (`basket_overrides`), gespeichert wie jeder andere Run via `src.backtest.runs.save_run`.

Gesucht wird bewusst NUR über `entry_z`, `exit_z`, `max_holding_half_life_multiplier` — siehe Modul-Docstring von `src/backtest/tuning.py` für die Data-Snooping-Begründung (Filter-/Fensterparameter sind theoriegeleitete Konventionen, keine Stellschrauben). Abgeleitet von `_template.ipynb` — siehe `README.md` für die Notebook-Konvention.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import time
import pandas as pd
from IPython.display import display

from src.data import load_ohlcv_panel, load_symbol_map
from src.backtest.cliques import CliqueTimeline
from src.backtest.config import BacktestConfig, BasketParams, split_calibration_backtest
from src.backtest.engine import BacktestEngine
from src.backtest import metrics as bt_metrics
from src.backtest import plots as bt_plots
from src.backtest import runs as bt_runs
from src.backtest import tuning as bt_tuning

print("Imports OK")

Imports OK


## 1. Run-Konfiguration

Der einzige Ort, den ein von diesem Template abgeleitetes Tuning-Notebook üblicherweise ändert. `BASE_PARAMS` legt alle NICHT gesuchten Parameter fest (Filter, Kosten, Positionslimits, ...) — sie bleiben für alle Trials und Baskets fix.

In [3]:
RUN_LABEL = "TEMPLATE_DO_NOT_RUN"  # beim Kopieren AENDERN, z.B. "tune_pairwise_basket1"
DESCRIPTION = ""

DATA_START = pd.Timestamp("2024-07-23", tz="UTC")   # frühestmögliches Datum im OHLCV-Panel
DATA_END = pd.Timestamp("2026-07-22", tz="UTC")     # spätestmögliches Datum
BACKTEST_FRACTION = 0.2  # gleiches Muster wie in _template.ipynb / 01_forward_return_screening.ipynb

BASKETS = ("Basket 1", "Basket 2", "Basket 3", "Basket 4")
CLIQUE_VARIANT = "raw_24h"  # raw|residual x 6h|12h|24h, siehe data/processed/cliques/
STRATEGY_NAME = "pairwise_zscore"  # oder "cluster_johansen"

INITIAL_CAPITAL = 100_000.0

# Walk-Forward-Suche (siehe src/backtest/tuning.py fuer die volle Methodik)
FOLD_DAYS = 30          # Laenge/Schrittweite jedes nicht-ueberlappenden Folds
N_TRIALS = 30           # NEUE Trials in diesem Aufruf (Study wird bei gleichem RUN_LABEL fortgesetzt)
MIN_TRADES_PER_FOLD = 1.0
SEED = 42

In [4]:
BASE_PARAMS = BasketParams(
    # entry_z/exit_z/max_holding_half_life_multiplier werden von der Suche ueberschrieben --
    # die hier gesetzten Werte sind nur der Ausgangspunkt fuer den finalen Backtest, falls
    # ein Basket keinen einzigen gueltigen Trial liefert (siehe Abschnitt 5).
    entry_z=2.0, exit_z=0.5, stop_z=4.0,
    formation_window_candles=288 * 7,
    max_candidate_pairs=3, pair_prefilter_top_n=15,
    min_half_life_candles=6, max_half_life_candles=288 * 2,
    max_concurrent_positions=3,
    fee_bps=10.0, slippage_bps=2.0,
)

## 2. Daten laden

In [5]:
id_to_symbol, symbol_to_id = load_symbol_map()

# Panel muss VOR DATA_START genug Historie fuer formation_window_candles haben
# (Warm-up-Guard G7 in engine.py) -- hier grosszuegig ab DATA_START selbst geladen.
panel = load_ohlcv_panel(start=DATA_START, end=DATA_END, fields=("open", "high", "low", "close"))
print(f"Panel: {panel['close'].shape[0]:,} Kerzen x {panel['close'].shape[1]} Symbole")

Panel: 209,953 Kerzen x 47 Symbole


In [6]:
timeline = CliqueTimeline.from_cache(
    variant=CLIQUE_VARIANT, id_to_symbol=id_to_symbol, baskets=list(BASKETS),
    start=DATA_START, end=DATA_END,
)
print(timeline.describe())

Discovery-Intervall (Median über Baskets): 1 days 00:00:00
  Basket 1: 531 Snapshots, 2024-07-29 00:00:00+00:00 → 2026-07-22 00:00:00+00:00, Intervall 1 days 00:00:00
  Basket 2: 605 Snapshots, 2024-07-29 00:00:00+00:00 → 2026-07-22 00:00:00+00:00, Intervall 1 days 00:00:00
  Basket 3: 459 Snapshots, 2024-08-05 00:00:00+00:00 → 2026-06-24 00:00:00+00:00, Intervall 1 days 00:00:00
  Basket 4: 204 Snapshots, 2024-08-06 00:00:00+00:00 → 2026-06-10 00:00:00+00:00, Intervall 1 days 00:00:00


## 3. Calibration/Backtest-Split

Die Walk-Forward-Suche läuft ausschliesslich innerhalb `CALIBRATION_RANGE` (strukturell erzwungen von `generate_walkforward_folds`, siehe `src/backtest/tuning.py`) — `BACKTEST_RANGE` bleibt für den finalen Out-of-Sample-Lauf in Abschnitt 5 unberührt, exakt derselbe Split wie in jedem anderen `04_backtest`-Notebook.

In [7]:
CALIBRATION_RANGE, BACKTEST_RANGE = split_calibration_backtest(DATA_START, DATA_END, fraction=BACKTEST_FRACTION)
assert CALIBRATION_RANGE.end < BACKTEST_RANGE.start  # keine Ueberlappung, hart geprueft
print(f"Calibration (Tuning):  {CALIBRATION_RANGE.start.date()} -> {CALIBRATION_RANGE.end.date()}")
print(f"Backtest (Out-of-Sample): {BACKTEST_RANGE.start.date()} -> {BACKTEST_RANGE.end.date()}")

if RUN_LABEL == "TEMPLATE_DO_NOT_RUN":
    raise SystemExit(
        "Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren "
        "(cp 05_tuning.ipynb NN_tune_kurzname.ipynb) und RUN_LABEL in Abschnitt 1 aendern, "
        "bevor die Suche ausgefuehrt wird."
    )

Calibration (Tuning):  2024-07-23 -> 2026-02-26
Backtest (Out-of-Sample): 2026-02-27 -> 2026-07-22


SystemExit: Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren (cp 05_tuning.ipynb NN_tune_kurzname.ipynb) und RUN_LABEL in Abschnitt 1 aendern, bevor die Suche ausgefuehrt wird.

c:\Users\ilija\.conda\envs\tda-trading\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 4. Walk-Forward-Suche (pro Basket)

`tune_all_baskets` läuft `tune_basket_params` für jeden Basket in Folge, mit versetztem Seed (siehe Docstring in `tuning.py` für die Begründung) und isoliert (`baskets=(basket,)` pro Fold-Engine-Lauf).

In [ ]:
t0 = time.time()
tuning_results = bt_tuning.tune_all_baskets(
    baskets=BASKETS, panel=panel, timeline=timeline, base_params=BASE_PARAMS,
    calibration_range=CALIBRATION_RANGE, strategy_name=STRATEGY_NAME,
    initial_capital=INITIAL_CAPITAL, fold_days=FOLD_DAYS, n_trials=N_TRIALS,
    min_trades_per_fold=MIN_TRADES_PER_FOLD, seed=SEED, study_name=None,
)
tuning_runtime_seconds = time.time() - t0
print(f"Tuning-Laufzeit: {tuning_runtime_seconds:.1f}s")

NameError: name 'time' is not defined

In [ ]:
BASKET_OVERRIDES = bt_tuning.build_basket_overrides(tuning_results)
display(pd.DataFrame(BASKET_OVERRIDES).T)

## 5. Tuning-Ergebnisse speichern

Best-Params + volle `trials_df` je Basket unter `data/processed/tuning/<run_id>/` (analog zu `src.backtest.runs.save_run`, siehe `src.backtest.tuning.save_tuning_results`) — unabhängig vom finalen Backtest-Run in Abschnitt 7, damit die Suche selbst nachvollziehbar bleibt (z.B. wie stabil `best_value` über die Trials war).

In [ ]:
tuning_run_dir = bt_tuning.save_tuning_results(tuning_results, label=RUN_LABEL)
print(f"Tuning-Ergebnisse gespeichert: {tuning_run_dir}")

for basket, result in tuning_results.items():
    print(f"\n{basket}: bestes Objective={result.study.best_value:.4f} ueber {len(result.trials_df)} Trials, {len(result.folds)} Folds")
    display(result.trials_df.sort_values("objective", ascending=False).head(5))

## 6. Finaler Out-of-Sample-Backtest mit getunten Parametern

Exakt derselbe Ablauf wie in jedem anderen `04_backtest`-Notebook (siehe `_template.ipynb`), nur mit `basket_overrides=BASKET_OVERRIDES` aus der Suche statt Hand-getunten Werten.

In [ ]:
config = BacktestConfig(
    label=RUN_LABEL, description=DESCRIPTION or f"Walk-Forward-getunt ({N_TRIALS} Trials/Basket, {FOLD_DAYS}d Folds)",
    calibration_range=CALIBRATION_RANGE, backtest_range=BACKTEST_RANGE,
    baskets=BASKETS, clique_variant=CLIQUE_VARIANT, strategy_name=STRATEGY_NAME,
    default_params=BASE_PARAMS, basket_overrides=BASKET_OVERRIDES,
    initial_capital=INITIAL_CAPITAL,
)

engine = BacktestEngine(config=config, panel=panel, timeline=timeline)
print(engine.describe_guards())

In [ ]:
t0 = time.time()
result = engine.run(progress=True)
runtime_seconds = time.time() - t0
print(f"Laufzeit: {runtime_seconds:.1f}s | {len(result.trades)} Trades | Warnungen: {len(engine.warnings)}")

## 7. Metriken

In [ ]:
BENCHMARK_SYMBOL = config.benchmark_symbol
benchmark_close = panel["close"][BENCHMARK_SYMBOL] if BENCHMARK_SYMBOL in panel["close"].columns else None

result.metrics = bt_metrics.compute_all_metrics(
    equity_curve=result.equity_curve, trades=result.trades,
    benchmark_close=benchmark_close, config=config,
)
result.metrics_by_basket = bt_metrics.metrics_by_basket(result.trades, config)
result.meta["runtime_seconds"] = runtime_seconds
result.meta["tuning_run_dir"] = str(tuning_run_dir)

display(result.summary())
display(result.metrics_by_basket)

## 8. Equity-Kurve & Drawdown

In [ ]:
bt_plots.plot_equity_and_drawdown(result, benchmark_close=benchmark_close)

## 9. Run speichern & mit früheren Runs vergleichen

In [ ]:
run_dir = bt_runs.save_run(result)
print(f"Gespeichert: {result.run_id}")
print(run_dir)

In [ ]:
display(bt_runs.list_runs(refresh_index=True).tail(10))

In [ ]:
# Konkrete Runs zum Vergleich eintragen, z.B. getunt vs. hand-parametrisiert:
# bt_runs.compare_runs([result.run_id, "<anderer-run-id>"])

## 10. Interpretation

_Freitext: wie stabil war `best_value` über die Trials (siehe `trials_df` in Abschnitt 5)? Wich der finale Out-of-Sample-Backtest stark von der Walk-Forward-Robustheitsschätzung ab (Overfitting-Indiz)? Welche Baskets profitierten am meisten vom Tuning gegenüber Hand-Parametern?_